## Thư viện

Hàm và hằng dùng chung, chép **nguyên văn** (không còn notebook `00_common`). Notebook khác có bản sao của cùng hàm — sửa ở đây thì sửa cả các bản kia cho khớp: bảng *Hàm có bản sao ở đâu* trong `README.md`. Phần riêng của notebook này bắt đầu sau mục này.

In [ ]:
# --- 1. Nền: đường dẫn, lược đồ, mã hoá ---
# Xác định gốc dự án từ thư mục làm việc của kernel Jupyter.
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "Code").is_dir() and (p / "Data").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ bên trong dự án SDC")
print("Gốc dự án:", ROOT)

In [ ]:
# --- Đường dẫn dự án ---
from pathlib import Path


DATA = ROOT / "Data"
FEAT_DIR = DATA / "features"
SESSIONS_PATH = DATA / "sessions_verified.parquet"

# pcap tự thu (điện thoại, laptop... trong mạng nhà). Tên thư mục khác nhau giữa server
# (`data_pcap`) và máy local (`data_pcap_capture`) — một chỗ quyết định cho mọi bước.
CAPTURE_DIR = next((DATA / name for name in ("data_pcap", "data_pcap_capture")
                    if (DATA / name).is_dir()), DATA / "data_pcap")

In [ ]:
# --- Lược đồ dữ liệu — head, nhóm feature, chia tập ---
import pandas as pd


MISSING = "<missing>"

META_COLS = ["canonical_device", "scenario", "capture_file", "date", "mac", "n_sources"]

# Ba head, đúng thứ tự output của sản phẩm: hãng / loại / model sản phẩm.
#
# Head thứ ba từng tên là `family` (contract ≤ 2.1). Đổi thành `model` cho khớp ô MODEL
# trong spec, nhưng chỉ là ĐỔI TÊN — nội dung nhãn không đổi, và độ mịn của nó vẫn là
# "model sản phẩm ở mức phân biệt được bằng telemetry": `Amazon Echo Dot` là một model,
# còn `Tuya Plug` vẫn gộp 13 thiết bị ODM không tách được bằng DHCP/DNS/TLS. Đổi tên
# không làm chúng tách ra được; xem `Data/device_labels_verified.csv` cột review_note.
MODEL_HEAD = "model"
LEGACY_MODEL_HEAD = "family"
HEAD_ALIASES = {LEGACY_MODEL_HEAD: MODEL_HEAD}
LABEL_COLS = ["make", "type", MODEL_HEAD]
SOURCES = ["dhcp", "dns", "mdns", "tls"]

# Cột dạng chuỗi token -> TF-IDF. Phần còn lại dạng chuỗi -> categorical nguyên giá trị.
TEXT_COLS = ["dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens"]

# Feature biến mất khi DNS được mã hoá (DoH/DoT) và SNI bị ẩn (ECH). Đây là lý do thật
# để đo bản không có chúng — không phải vì sợ leak: lúc chạy thật router vẫn đọc được
# DNS query và SNI, nên chúng là bằng chứng hợp lệ.
ENCRYPTED_RISK = ["dns_tokens", "tls_sni_tokens"]

# Field driver KHÔNG cung cấp — không feature nào được suy ra từ chúng
FORBIDDEN = ["src_ip", "dst_ip", "dst_port", "ts", "iat", "pktsize", "bytes", "rate"]


# Cot co-mat cua tung nguon. Suy tu `SOURCES` de danh sach nguon chi co MOT ban.
SOURCE_FLAGS = tuple(f"has_{source}" for source in SOURCES)


def feature_sets(df):
    """Các nhóm feature, suy từ chính các cột của dataframe."""
    feats = [c for c in df.columns if c not in META_COLS + LABEL_COLS]
    text = [c for c in TEXT_COLS if c in feats]
    cat = [c for c in feats if c not in text and not pd.api.types.is_numeric_dtype(df[c])]
    num = [c for c in feats if c not in text and c not in cat]
    return {
        "all": feats,
        "text": text,
        "cat": cat,
        "num": num,
        "source": {s: [c for c in feats if c.startswith(s)] for s in SOURCES},
        "no_content": [c for c in feats if c not in ENCRYPTED_RISK],
    }


def assert_serveable(cols):
    """Fail nếu có feature vượt quá telemetry driver cung cấp."""
    bad = [c for c in cols if any(f in c for f in FORBIDDEN)]
    assert not bad, f"Feature không phục vụ được lúc chạy thật: {bad}"
    orphan = [c for c in cols if not c.startswith(("dhcp_", "dns_", "mdns_", "tls_", "has_"))]
    assert not orphan, f"Feature không truy được về nguồn nào: {orphan}"
    return True


def normalize_heads(df):
    """Đổi tên cột nhãn cũ sang tên head hiện hành (`family` -> `model`).

    Làm ở đây, một chỗ, thay vì ghi lại parquet: notebook 03_build_dataset và các file dữ liệu đã sinh
    ra từ trước vẫn dùng được, mà phần còn lại của code chỉ biết đúng một tên. Cột mới
    đã có sẵn thì không đụng vào — dữ liệu mới luôn thắng.
    """
    rename = {old: new for old, new in HEAD_ALIASES.items()
              if old in df.columns and new not in df.columns}
    # Có cả hai cột: cột mới là nhãn đang dùng, cột cũ là tàn dư của file dựng trước lần
    # đổi tên. Bỏ hẳn cột cũ — để lại thì nó không phải nhãn mà cũng không phải feature,
    # và `assert_serveable` sẽ bắt nó như một feature không truy được về nguồn nào.
    stale = [old for old, new in HEAD_ALIASES.items()
             if old in df.columns and new in df.columns]
    if rename or stale:
        df = df.rename(columns=rename).drop(columns=stale)
    return df


def load_sessions(path=SESSIONS_PATH):
    """Nạp feature table mức session kèm nhóm feature đã kiểm tra."""
    assert path.exists(), f"Không tìm thấy {path} — chạy 03_build_dataset.ipynb trước"
    df = normalize_heads(pd.read_parquet(path))
    fs = feature_sets(df)
    assert_serveable(fs["all"])
    return df, fs

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
import hashlib
import re
import struct
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


try:  # Packet parsing is not needed by record-only production deployments.
    from scapy.all import BOOTP, DHCP, DNS, Ether, IP, IPv6, PcapReader, Raw, TCP, UDP
except ImportError:  # pragma: no cover - exercised only in minimal production images
    BOOTP = DHCP = DNS = Ether = IP = IPv6 = PcapReader = Raw = TCP = UDP = None


MAX_TOKENS = 40
MAX_TLS_STREAM_BYTES = 256 * 1024
TOKEN_SPLIT = re.compile(r"[.\-]")

RAW_FIELDS = {
    "dhcp": ("opt55", "opt60"),
    "dns": ("qname",),
    "mdns": ("qname",),
    "tls": ("version", "ciphers", "alpn", "sni"),
}
# Cột cờ có-mặt nằm ở `sdc_schema` vì đó là lược đồ chung, không riêng của bước parse.
# Gác này bắt trường hợp hai nơi kể tên nguồn lệch nhau — im lặng thì `n_sources` sai.
assert tuple(RAW_FIELDS) == tuple(SOURCES), (
    f"RAW_FIELDS {tuple(RAW_FIELDS)} lệch SOURCES {tuple(SOURCES)}"
)

DHCP_MSG_TYPES = {
    1: "DISCOVER", 2: "OFFER", 3: "REQUEST", 4: "DECLINE",
    5: "ACK", 6: "NAK", 7: "RELEASE", 8: "INFORM",
}
DNS_QTYPES = {1: "A", 28: "AAAA", 12: "PTR", 16: "TXT", 33: "SRV",
              5: "CNAME", 255: "ANY"}
DNS_PORTS = (53, 5353)


def _require_scapy():
    if PcapReader is None:
        raise RuntimeError("Packet extraction requires Scapy; install it with 'pip install scapy'")


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
        return bool(missing) if isinstance(missing, (bool, np.bool_)) else False
    except (TypeError, ValueError):
        return False


def _text(value, *, lowercase=False):
    if _is_missing(value):
        return None
    if isinstance(value, bytes):
        value = value.decode(errors="replace")
    value = str(value).strip()
    if not value:
        return None
    return value.lower() if lowercase else value


def normalize_mac(value):
    """Return a lowercase colon-separated six-byte MAC, or ``None``."""
    if isinstance(value, bytes):
        raw = value[:6].hex()
    else:
        raw = re.sub(r"[^0-9a-f]", "", str(value or "").lower())[:12]
    if len(raw) != 12 or raw == "0" * 12:
        return None
    return ":".join(raw[index:index + 2] for index in range(0, 12, 2))


def packet_source_mac(pkt):
    _require_scapy()
    if not pkt.haslayer(Ether):
        return None
    return normalize_mac(pkt[Ether].src)


def bootp_client_mac(pkt):
    _require_scapy()
    if not pkt.haslayer(BOOTP):
        return None
    return normalize_mac(bytes(pkt[BOOTP].chaddr or b"")[:6])


def _dhcp_options_dict(dhcp_layer):
    return {option[0]: option[1] for option in dhcp_layer.options
            if isinstance(option, tuple) and len(option) >= 2}


def extract_dhcp(pkt, ts=None):
    """Extract DHCP client identity fields in canonical and training-compatible names."""
    _require_scapy()
    if not pkt.haslayer(DHCP):
        return None
    options = _dhcp_options_dict(pkt[DHCP])
    hostname = _text(options.get("hostname"), lowercase=True)
    # Preserve VCI bytes for compatibility with exact L1 keys. The text encoder
    # performs its own lowercase normalization for L2.
    vendor_class_id = _text(options.get("vendor_class_id"))

    request_list = options.get("param_req_list")
    if isinstance(request_list, bytes):
        request_list = list(request_list)
    if request_list is not None and not isinstance(request_list, (list, tuple)):
        request_list = [request_list]
    param_req_list = (",".join(str(int(option)) for option in request_list)
                      if request_list else None)
    if param_req_list is None and vendor_class_id is None:
        return None

    msg_type_code = options.get("message-type")
    try:
        msg_type_code = int(msg_type_code)
    except (TypeError, ValueError):
        pass
    timestamp = float(pkt.time) if ts is None and hasattr(pkt, "time") else ts
    client_mac = bootp_client_mac(pkt)
    return {
        "proto": "dhcp",
        "opt55": param_req_list,
        "opt60": vendor_class_id,
        "ts": timestamp,
        "src_mac": packet_source_mac(pkt),
        "client_mac": client_mac,
        "msg_type_code": msg_type_code,
        "msg_type": DHCP_MSG_TYPES.get(msg_type_code, msg_type_code),
        "hostname": hostname,
        "vendor_class_id": vendor_class_id,
        "param_req_list": param_req_list,
    }


def _network_addresses(pkt):
    if IP is not None and pkt.haslayer(IP):
        return str(pkt[IP].src), str(pkt[IP].dst)
    if IPv6 is not None and pkt.haslayer(IPv6):
        return str(pkt[IPv6].src), str(pkt[IPv6].dst)
    return None, None

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def extract_dns(pkt, ts=None):
    """Extract DNS and mDNS queries; responses are never feature evidence."""
    _require_scapy()
    if not pkt.haslayer(DNS):
        return []
    transport = pkt[UDP] if pkt.haslayer(UDP) else (pkt[TCP] if pkt.haslayer(TCP) else None)
    if transport is None:
        return []
    ports = (int(transport.sport), int(transport.dport))
    if not any(port in DNS_PORTS for port in ports):
        return []

    dns = pkt[DNS]
    if bool(dns.qr):
        return []
    is_mdns = 5353 in ports
    src_ip, dst_ip = _network_addresses(pkt)
    timestamp = float(pkt.time) if ts is None and hasattr(pkt, "time") else ts
    rows = []
    questions = list(dns.qd or []) if isinstance(dns.qd, (list, tuple)) else [dns.qd]
    for question in questions:
        if question is None:
            continue
        qname = _text(question.qname, lowercase=True)
        if not qname:
            continue
        qname = qname.rstrip(".")
        rows.append({
            "proto": "mdns" if is_mdns else "dns",
            "qname": qname,
            "ts": timestamp,
            "src_ip": src_ip,
            "dst_ip": dst_ip,
            "is_mdns": is_mdns,
            "is_response": False,
            "qry_name": qname,
            "qry_type": DNS_QTYPES.get(question.qtype, question.qtype),
            "rcode": None,
        })
    return rows


def is_grease(value):
    """RFC 8701 GREASE values have both bytes equal and a low nibble of 0xA."""
    try:
        value = int(value)
    except (TypeError, ValueError):
        return False
    return 0 <= value <= 0xFFFF and (value & 0x0F0F) == 0x0A0A


def normalize_cipher_suites(value):
    if _is_missing(value):
        return MISSING
    values = value if isinstance(value, (list, tuple)) else re.findall(r"\d+", str(value))
    normalized = []
    for item in values:
        try:
            number = int(item)
        except (TypeError, ValueError):
            continue
        if not is_grease(number):
            normalized.append(str(number))
    return ",".join(normalized) if normalized else MISSING


def normalize_tls_version(value):
    if _is_missing(value):
        return MISSING
    text = str(value).strip().lower()
    if text.startswith("0x"):
        text = text[2:]
    if re.fullmatch(r"\d{1,4}", text):
        return text.zfill(4)
    return MISSING


def parse_tls_client_hello(raw):
    """Parse a complete TLS ClientHello record and remove GREASE from its fingerprint."""
    try:
        raw = bytes(raw)
        if len(raw) < 9 or raw[0] != 0x16:
            return None
        record_length = struct.unpack("!H", raw[3:5])[0]
        if record_length > MAX_TLS_STREAM_BYTES or len(raw) < 5 + record_length:
            return None
        handshake = raw[5:5 + record_length]
        if len(handshake) < 4 or handshake[0] != 0x01:
            return None
        handshake_length = int.from_bytes(handshake[1:4], "big")
        if len(handshake) < 4 + handshake_length:
            return None
        body = handshake[4:4 + handshake_length]

        position = 0
        if len(body) < 35:
            return None
        version = body[position:position + 2].hex()
        position += 34  # version + random
        session_length = body[position]
        position += 1
        if position + session_length + 2 > len(body):
            return None
        position += session_length

        cipher_length = struct.unpack("!H", body[position:position + 2])[0]
        position += 2
        if cipher_length % 2 or position + cipher_length > len(body):
            return None
        cipher_suites = [struct.unpack("!H", body[offset:offset + 2])[0]
                         for offset in range(position, position + cipher_length, 2)]
        cipher_suites = [value for value in cipher_suites if not is_grease(value)]
        position += cipher_length

        if position >= len(body):
            return None
        compression_length = body[position]
        position += 1
        if position + compression_length > len(body):
            return None
        position += compression_length

        sni, alpn = None, []
        if position + 2 <= len(body):
            extensions_length = struct.unpack("!H", body[position:position + 2])[0]
            position += 2
            extensions_end = position + extensions_length
            if extensions_end > len(body):
                return None
            while position + 4 <= extensions_end:
                extension_type, extension_length = struct.unpack(
                    "!HH", body[position:position + 4]
                )
                position += 4
                if position + extension_length > extensions_end:
                    return None
                data = body[position:position + extension_length]
                position += extension_length

                if extension_type == 0 and len(data) >= 5:
                    names_length = struct.unpack("!H", data[:2])[0]
                    cursor = 2
                    names_end = min(len(data), 2 + names_length)
                    while cursor + 3 <= names_end:
                        name_type = data[cursor]
                        name_length = struct.unpack("!H", data[cursor + 1:cursor + 3])[0]
                        cursor += 3
                        if cursor + name_length > names_end:
                            break
                        if name_type == 0:
                            sni = _text(data[cursor:cursor + name_length], lowercase=True)
                            break
                        cursor += name_length
                elif extension_type == 16 and len(data) >= 2:
                    protocols_length = struct.unpack("!H", data[:2])[0]
                    cursor = 2
                    protocols_end = min(len(data), 2 + protocols_length)
                    while cursor < protocols_end:
                        protocol_length = data[cursor]
                        cursor += 1
                        if cursor + protocol_length > protocols_end:
                            break
                        protocol = _text(data[cursor:cursor + protocol_length], lowercase=True)
                        if protocol:
                            alpn.append(protocol)
                        cursor += protocol_length

        ciphers = ",".join(str(value) for value in cipher_suites)
        if not ciphers:
            return None
        alpn_value = ",".join(alpn) if alpn else None
        return {
            "proto": "tls",
            "version": version,
            "ciphers": ciphers,
            "alpn": alpn_value,
            "sni": sni,
            "tls_version": version,
            "cipher_suites": ciphers,
            "n_cipher_suites": len(cipher_suites),
        }
    except (IndexError, TypeError, ValueError, struct.error):
        return None


def _tls_packet_metadata(pkt, ts):
    src_ip, dst_ip = _network_addresses(pkt)
    return {
        "ts": float(pkt.time) if ts is None and hasattr(pkt, "time") else ts,
        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "dst_port": int(pkt[TCP].dport),
    }

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
class TLSStreamReassembler:
    """Small bounded TCP reassembler for outbound TLS ClientHello records."""

    def __init__(self, max_stream_bytes=MAX_TLS_STREAM_BYTES):
        self.max_stream_bytes = max_stream_bytes
        self._segments = {}
        self._emitted = set()

    @staticmethod
    def _flow_key(pkt):
        src_ip, dst_ip = _network_addresses(pkt)
        if src_ip is None:
            src_ip, dst_ip = packet_source_mac(pkt), None
        tcp = pkt[TCP]
        return src_ip, dst_ip, int(tcp.sport), int(tcp.dport)

    @staticmethod
    def _assemble(segments):
        chunks = []
        cursor = None
        for sequence, payload in sorted(segments.items()):
            if cursor is None:
                cursor = sequence
            if sequence > cursor:
                break
            offset = max(0, cursor - sequence)
            if offset < len(payload):
                chunks.append(payload[offset:])
                cursor += len(payload) - offset
        return b"".join(chunks)

    def feed(self, pkt, ts=None):
        _require_scapy()
        if not (pkt.haslayer(TCP) and pkt.haslayer(Raw)):
            return []
        payload = bytes(pkt[Raw].load)
        if not payload:
            return []
        flow = self._flow_key(pkt)
        segments = self._segments.setdefault(flow, {})
        sequence = int(pkt[TCP].seq)
        if len(payload) > len(segments.get(sequence, b"")):
            segments[sequence] = payload
        if sum(map(len, segments.values())) > self.max_stream_bytes:
            self._segments.pop(flow, None)
            return []

        stream = self._assemble(segments)
        found = []
        offset = 0
        while offset + 5 <= len(stream):
            if stream[offset] != 0x16 or stream[offset + 1] != 0x03:
                offset += 1
                continue
            record_length = struct.unpack("!H", stream[offset + 3:offset + 5])[0]
            total = 5 + record_length
            if record_length > self.max_stream_bytes:
                offset += 1
                continue
            if offset + total > len(stream):
                break
            record = stream[offset:offset + total]
            parsed = parse_tls_client_hello(record)
            if parsed is not None:
                digest = hashlib.sha256(record).digest()
                marker = (flow, digest)
                if marker not in self._emitted:
                    self._emitted.add(marker)
                    parsed.update(_tls_packet_metadata(pkt, ts))
                    found.append(parsed)
            offset += total
        return found


def domain_tokens(names, max_tokens=MAX_TOKENS):
    """Domain names to unique lowercase tokens, preserving first-seen order."""
    tokens = []
    for name in names:
        if _is_missing(name):
            continue
        normalized = str(name).lower().rstrip(".")
        tokens.extend(token for token in TOKEN_SPLIT.split(normalized) if token)
    unique = list(dict.fromkeys(tokens))[:max_tokens]
    return " ".join(unique) if unique else MISSING


def _mode(values):
    clean = [str(value) for value in values
             if not _is_missing(value) and str(value).strip() not in ("", MISSING)]
    if not clean:
        return MISSING
    return pd.Series(clean).mode().iat[0]


def _tls_fingerprint(record):
    version = normalize_tls_version(record.get("version", record.get("tls_version")))
    ciphers = normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
    alpn = _text(record.get("alpn"), lowercase=True) or ""
    if MISSING in (version, ciphers):
        return MISSING
    return f"{version}|{ciphers}|{alpn}"

In [ ]:
# --- Trích xuất feature từ pcap và bản ghi raw ---
def aggregate(records, feature_cols):
    """Aggregate canonical records for one device/window into one model feature row."""
    by_source = {source: [] for source in RAW_FIELDS}
    for record in records:
        if not isinstance(record, dict):
            raise TypeError("Each raw record must be a dictionary")
        proto = record.get("proto")
        if proto not in by_source:
            raise ValueError(f"Invalid proto {proto!r}; expected one of {sorted(RAW_FIELDS)}")
        by_source[proto].append(record)

    row = {}
    row["dhcp_prl"] = _mode(record.get("opt55", record.get("param_req_list"))
                             for record in by_source["dhcp"])
    row["dhcp_vci"] = _mode(record.get("opt60", record.get("vendor_class_id"))
                             for record in by_source["dhcp"])
    options = set()
    if row["dhcp_prl"] != MISSING:
        options = {int(value) for value in re.findall(r"\d+", row["dhcp_prl"])}
    for column in feature_cols:
        if column.startswith("dhcp_opt_"):
            row[column] = int(int(column.rsplit("_", 1)[1]) in options)
    row["dhcp_prl_len"] = len(options)

    row["dns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                         for record in by_source["dns"])
    row["mdns_tokens"] = domain_tokens(record.get("qname", record.get("qry_name"))
                                          for record in by_source["mdns"])

    tls_records = by_source["tls"]
    row["tls_fp"] = _mode(_tls_fingerprint(record) for record in tls_records)
    row["tls_version"] = _mode(
        normalize_tls_version(record.get("version", record.get("tls_version")))
        for record in tls_records
    )
    row["tls_alpn"] = _mode(_text(record.get("alpn"), lowercase=True)
                              for record in tls_records)
    row["tls_ciphers"] = _mode(
        normalize_cipher_suites(record.get("ciphers", record.get("cipher_suites")))
        for record in tls_records
    )
    row["tls_sni_tokens"] = domain_tokens(record.get("sni") for record in tls_records)

    for source in RAW_FIELDS:
        row[f"has_{source}"] = int(bool(by_source[source]))
    row["n_sources"] = sum(row[flag] for flag in SOURCE_FLAGS)

    numeric_prefixes = ("has_", "dhcp_opt_")
    for column in feature_cols:
        if column not in row:
            row[column] = (0 if column.startswith(numeric_prefixes)
                           or column == "dhcp_prl_len" else MISSING)
    return row

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

try:
    import scapy
    print("scapy  :", scapy.VERSION)
except ImportError:
    raise SystemExit("Cần scapy để đọc pcap: pip install scapy")


## 1. Liệt kê pcap mới

Bỏ qua file rác AppleDouble (`._*.pcap`), cùng quy tắc với `01_extract_cic.ipynb`.

In [ ]:
from scapy.all import PcapReader, Ether

PCAP_DIR = CAPTURE_DIR
OUT_DIR = FEAT_DIR  # Data/features, đã định nghĩa ở 03
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PCAP_DIR.exists(), f"Không tìm thấy {PCAP_DIR}"

pcap_files = sorted(p for p in PCAP_DIR.glob("*.pcap") if not p.name.startswith("._"))
print(f"{len(pcap_files)} file pcap")
for p in pcap_files:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")


## 1bis. Bang ten thiet bi (hostname) theo MAC

`Data/data_pcap/device_map.csv` (hostname, mac, ip) lay tu bang DHCP lease +
ARP cua router luc bat goi (`host_name_join.png`, `MAC.png`). Dung de gan them
cot `device` de doc ben canh `mac` tren moi output - khong thay `mac` vi do
moi la khoa session that. MAC la (khong co trong bang) duoc gan nhan
`Unknown-<mac>` thay vi bo trong.

Khi co danh sach thiet bi moi, chi can sua lai `device_map.csv` roi chay lai
notebook - khong can sua code.


In [ ]:
import csv

DEVICE_MAP_CSV = PCAP_DIR / "device_map.csv"


def load_device_map(path):
    mapping = {}
    if not path.exists():
        return mapping
    with open(path, newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            mac = normalize_mac(row["mac"])
            if mac:
                mapping[mac] = row["hostname"].strip()
    return mapping


mac_to_device = load_device_map(DEVICE_MAP_CSV)


def device_for_mac(mac):
    if mac is None:
        return "Unknown"
    return mac_to_device.get(mac, f"Unknown-{mac.replace(':', '')}")


print(f"{len(mac_to_device)} thiet bi trong {DEVICE_MAP_CSV.name}")


## 2. Trích xuất theo MAC nguồn

Mỗi file pcap = một lần bắt gói trên cả mạng, nhiều thiết bị trộn chung. Không có
`device_mac` biết trước như `POWER`/IoT Sentinel, nên xử lý toàn bộ gói trong file
(không lọc theo `packet_from_device`) và tự gom theo MAC:

- **DHCP** — gắn theo `client_mac` (`BOOTP.chaddr`, qua `extract_dhcp`) nếu có, để cả gói
  OFFER/ACK do server gửi vẫn quy đúng về thiết bị xin cấp IP; nếu thiếu thì rơi về
  `Ether.src` của chính gói đó.
- **DNS/mDNS/TLS** — luôn do chính thiết bị phát ra (`extract_dns` chỉ lấy query,
  ClientHello luôn từ client), nên gắn thẳng theo `Ether.src`.

`(mac, capture_file)` là đơn vị session — giữ nguyên record thô song song (không chỉ
bảng phẳng) để mục 4 gọi `aggregate()` mà không phải suy ngược từ CSV.

In [ ]:
from collections import defaultdict

from tqdm.auto import tqdm

dhcp_rows, dns_rows, mdns_rows, tls_rows = [], [], [], []
summary_counts = defaultdict(lambda: {"n_dhcp": 0, "n_dns": 0, "n_mdns": 0, "n_tls": 0})
records_by_session = defaultdict(list)  # (mac, capture_file) -> [record, ...] cho aggregate()

for pcap_path in tqdm(pcap_files, desc="capture files"):
    tls_streams = {}  # mac -> TLSStreamReassembler, tách riêng theo thiết bị

    with PcapReader(str(pcap_path)) as reader:
        for pkt in reader:
            if not pkt.haslayer(Ether):
                continue
            ts = float(pkt.time)

            dhcp_row = extract_dhcp(pkt, ts)
            if dhcp_row:
                mac = dhcp_row.get("client_mac") or dhcp_row.get("src_mac")
                if mac:
                    row = {"mac": mac, "capture_file": pcap_path.name, **dhcp_row}
                    dhcp_rows.append(row)
                    records_by_session[(mac, pcap_path.name)].append(dhcp_row)
                    summary_counts[(mac, pcap_path.name)]["n_dhcp"] += 1

            mac = packet_source_mac(pkt)
            if mac is None:
                continue

            for dns_row in extract_dns(pkt, ts):
                row = {"mac": mac, "capture_file": pcap_path.name, **dns_row}
                records_by_session[(mac, pcap_path.name)].append(dns_row)
                if dns_row["is_mdns"]:
                    mdns_rows.append(row)
                    summary_counts[(mac, pcap_path.name)]["n_mdns"] += 1
                else:
                    dns_rows.append(row)
                    summary_counts[(mac, pcap_path.name)]["n_dns"] += 1

            tls_stream = tls_streams.setdefault(mac, TLSStreamReassembler())
            for tls_row in tls_stream.feed(pkt, ts):
                tls_rows.append({"mac": mac, "capture_file": pcap_path.name, **tls_row})
                records_by_session[(mac, pcap_path.name)].append(tls_row)
                summary_counts[(mac, pcap_path.name)]["n_tls"] += 1

dhcp_df = pd.DataFrame(dhcp_rows)
dns_df = pd.DataFrame(dns_rows)
mdns_df = pd.DataFrame(mdns_rows)
tls_df = pd.DataFrame(tls_rows)
summary_df = pd.DataFrame(
    [{"mac": mac, "capture_file": capture_file, **counts}
     for (mac, capture_file), counts in summary_counts.items()]
)

print("dhcp_df:", dhcp_df.shape, "dns_df:", dns_df.shape,
      "mdns_df:", mdns_df.shape, "tls_df:", tls_df.shape)
print(f"{summary_df[['mac']].drop_duplicates().shape[0]} MAC nguồn, "
      f"{len(records_by_session)} session (mac, capture_file)")


Gan cot `device` (hostname) cho tung dataframe theo `mac`.

In [ ]:
for _df in (dhcp_df, dns_df, mdns_df, tls_df, summary_df):
    if not _df.empty:
        _df.insert(_df.columns.get_loc("mac") + 1, "device", _df["mac"].map(device_for_mac))


## 3. Lưu output raw

Ghi đè mỗi lần chạy — thư mục `data_pcap/` còn đang nhận thêm file.

In [ ]:
dhcp_df.to_csv(OUT_DIR / "dhcp_features_capture.csv", index=False)
dns_df.to_csv(OUT_DIR / "dns_features_capture.csv", index=False)
mdns_df.to_csv(OUT_DIR / "mdns_features_capture.csv", index=False)
tls_df.to_csv(OUT_DIR / "tls_features_capture.csv", index=False)
summary_df.to_csv(OUT_DIR / "capture_summary_capture.csv", index=False)

print("Đã lưu vào", OUT_DIR.resolve())
for f in ["dhcp_features_capture.csv", "dns_features_capture.csv", "mdns_features_capture.csv",
          "tls_features_capture.csv", "capture_summary_capture.csv"]:
    print(" -", f)


## 4. Gộp thành bảng 40 cột (sẵn sàng cho model)

Gọi đúng `aggregate()` ở mục Thư viện trên record thô đã giữ ở mục 2 —
không tự gộp tay, tránh lệch với lúc train/serve (xem `README.md` mục 2.7).

In [ ]:
frame, feature_groups = load_sessions(SESSIONS_PATH)
features = feature_groups["all"]

agg_rows = []
for (mac, capture_file), records in records_by_session.items():
    row = {"mac": mac, "capture_file": capture_file, **aggregate(records, features)}
    agg_rows.append(row)

session_df = pd.DataFrame(agg_rows)

session_df.insert(1, "device", session_df["mac"].map(device_for_mac))
session_df.to_csv(OUT_DIR / "session_features_capture.csv", index=False)

print(f"{len(session_df)} session x {len(features)} cột feature")
print("Đã lưu", OUT_DIR / "session_features_capture.csv")
display(session_df[["mac", "device", "capture_file", "has_dhcp", "has_dns", "has_mdns", "has_tls",
                     "n_sources", "dhcp_vci", "tls_sni_tokens"]])


## 5. Kiểm tra nhanh (preview)

Số nguồn có mặt theo MAC, cùng tinh thần mục 7 của `01_extract_cic.ipynb`.

In [ ]:
print("Phân bố số nguồn feature có mặt / (mac, capture_file):")
display(session_df["n_sources"].value_counts().sort_index())

print()
print("Chi tiết theo MAC:")
display(session_df.groupby(["mac", "device"])[["has_dhcp", "has_dns", "has_mdns", "has_tls"]].max())
